# Ghazal transliteration on A100: before and after fine-tuning

One notebook, no repository, no uploads. **Runtime → Change runtime type → A100 → Run all.**

Rekhta publishes every ghazal in three scripts: Nastaliq, Devanagari and Roman. This notebook fine-tunes a model to **transliterate verse between them, in all six directions**, and measures it before and after on poems it never saw, including two poets it never saw.

Transliteration is the task here because it is learned (not rule-based), useful, and testable on unseen poems. It's harder than it looks: Nastaliq omits short vowels, so going *from* Nastaliq means inferring vocalization, izafat and nasalization; going *to* Nastaliq means choosing between letters Devanagari and Roman don't distinguish (س / ص / ث, ز / ذ / ض / ظ).

Radif and qafiya are deterministic, so they're not trained. They're used as **rule-based checks**: after transliterating a matla, does it still have the same radif and qafiya?

**What you get:**
- character and word error rates per direction, before → after
- errors broken down by phenomenon (nukta, aspirates, nasalization, izafat, spelling choices, …)
- a learning curve: validation error against training examples seen (does more data help?)
- side-by-side examples and an automatic verdict

| `SIZE` | training shers | test poems / split | Sarvam-30B, A100 40 GB* | Qwen2.5-7B |
|---|---|---|---|---|
| `small` | 3,000 | 40 | ~1.5–2 h | ~25 min |
| `medium` | 10,000 | 100 | ~3–4 h | ~1 h |
| `large` | 30,000 | all | ~7–9 h | ~2.5 h |

*Including the one-time ~129 GB download. Estimates: the notebook prints measured times and ETAs as it runs. Start with `small`.

**Running Sarvam-30B:** start from a fresh runtime (**Runtime → Disconnect and delete runtime**), because Sarvam's custom code needs `transformers` 4.x, which the setup cell installs automatically. The preflight's **Sanity** line must be all `True`; if not, the notebook stops before the baseline. With `SAVE_TO_DRIVE` on, results land in `MyDrive/ghazal_tuning/` and Colab asks for Drive access once.

## 1. Settings

In [ ]:
MODEL = "qwen2.5-14b"    # @param ["sarvam-30b", "qwen2.5-7b", "qwen2.5-14b", "custom"]
CUSTOM_MODEL_ID = ""     # @param {type:"string"}
SIZE = "medium"          # @param ["small", "medium", "large"]

# Optional overrides (None = use the SIZE preset)
N_TRAIN = None           # training examples (one sher, one direction each)
EVAL_POEMS = None        # poems per test split
EPOCHS = None
LEARNING_RATE = 2e-4
LORA_R = 16

THINK_MODE = "auto"      # @param ["auto", "prefill", "allow"]
# auto: detect in preflight; if the model reasons before answering, an empty closed <think></think>
# is pre-filled for every prompt (base and tuned alike), so it answers directly. allow: let it think.

CORPUS_JSONL = ""        # @param {type:"string"}
# Empty = public Rekhta sample. Or your own export, one poem per line:
# {"id","poet","form","text":{"nastaliq":[...],"devanagari":[...],"roman":[...]},"poet_names":{...}}

SAVE_TO_DRIVE = True     # @param {type:"boolean"}
# True keeps predictions, adapter and reports on Google Drive (asks permission once). Re-running after a
# disconnect resumes finished evaluations.

FORCE_4BIT = False       # @param {type:"boolean"}
# On an 80 GB GPU, Sarvam-30B loads in bf16 (~64 GB). If training then runs out of memory,
# set this to True and Run all again: 4-bit uses ~20 GB.

SEED = 12345
TRANSFORMERS_VERSION = ""  # "" = auto: Sarvam-30B gets transformers 4.x (its custom code predates 5.x and
                           # produces garbage under 5.20); other models get the latest. Or set a spec, e.g. "==4.57.1".

PRESETS = {
    "small":  {"n_train": 3000,  "eval_poems": 40,   "epochs": 1, "probe_poems": 10},
    "medium": {"n_train": 10000, "eval_poems": 100,  "epochs": 1, "probe_poems": 15},
    "large":  {"n_train": 30000, "eval_poems": None, "epochs": 1, "probe_poems": 20},
}
P = dict(PRESETS[SIZE])
if N_TRAIN is not None: P["n_train"] = N_TRAIN
if EVAL_POEMS is not None: P["eval_poems"] = EVAL_POEMS
if EPOCHS is not None: P["epochs"] = EPOCHS

MODELS = {   # quantization is chosen from GPU memory in the next cell
    "sarvam-30b":  {"id": "sarvamai/sarvam-30b", "remote_code": True, "bf16_min_gb": 75, "gen_batch": 16,
                    "train_batch": 2, "grad_accum": 8, "chat_kwargs": {"enable_thinking": False}},
    "qwen2.5-7b":  {"id": "Qwen/Qwen2.5-7B-Instruct", "remote_code": False, "bf16_min_gb": 30, "gen_batch": 32,
                    "train_batch": 8, "grad_accum": 2, "chat_kwargs": {}},
    "qwen2.5-14b": {"id": "Qwen/Qwen2.5-14B-Instruct", "remote_code": False, "bf16_min_gb": 60, "gen_batch": 24,
                    "train_batch": 4, "grad_accum": 4, "chat_kwargs": {}},
    "custom":      {"id": CUSTOM_MODEL_ID, "remote_code": True, "bf16_min_gb": 60, "gen_batch": 16,
                    "train_batch": 4, "grad_accum": 4, "chat_kwargs": {}},
}
M = MODELS[MODEL]
ANSWER_TOKENS = 200      # one sher (two lines); Nastaliq is token-hungry
THINK_TOKENS = 768       # extra budget per answer, only when THINK_MODE ends up "allow"
MAX_SEQ_LEN = 512
print(f"Model {M['id']} · size {SIZE}: {P}")

## 2. Runtime

In [ ]:
import os, sys, shutil, subprocess, time, json, re, random, hashlib, math, gc, unicodedata, zipfile
os.environ["HF_HOME"] = "/content/hf_cache"          # local disk: the Sarvam checkpoint is ~129 GB
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100"
props = torch.cuda.get_device_properties(0)
GPU_GB = props.total_memory / 1e9
FREE_DISK_GB = shutil.disk_usage("/content").free / 1e9
QUANT = "4bit" if FORCE_4BIT or GPU_GB < M["bf16_min_gb"] else "none"
print(f"GPU {props.name} ({GPU_GB:.0f} GB) · free disk {FREE_DISK_GB:.0f} GB · loading {MODEL} as {QUANT}")
if MODEL == "sarvam-30b" and FREE_DISK_GB < 140:
    print("⚠️  Under 140 GB free disk: the Sarvam-30B checkpoint (~129 GB) may not fit. Try MODEL='qwen2.5-7b'.")

TF_SPEC = TRANSFORMERS_VERSION or (">=4.56,<5" if MODEL == "sarvam-30b" else ">=4.50")
V4 = "<5" in TF_SPEC or TF_SPEC.startswith("==4")
pkgs = ["accelerate>=0.33", "bitsandbytes>=0.43", "matplotlib", "pandas", "tqdm", "tabulate",
        f"transformers{TF_SPEC}", "peft>=0.13,<0.18" if V4 else "peft>=0.13"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
# Colab ships torchao 0.10; recent peft refuses to run next to torchao < 0.16 even though it isn't used here.
try:
    from importlib.metadata import version as _pkg_version
    if tuple(int(x) for x in _pkg_version("torchao").split(".")[:2]) < (0, 16):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        print("Removed an old torchao that conflicts with peft (not used by this notebook).")
except Exception:
    pass

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from difflib import SequenceMatcher
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # custom model code triggers many deprecation notices
warnings.filterwarnings("ignore", message=".*torch_dtype.*")
import transformers, peft
transformers.logging.set_verbosity_error()
print(f"torch {torch.__version__} · transformers {transformers.__version__} · peft {peft.__version__}")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_OUT = "/content/drive/MyDrive/ghazal_tuning"
else:
    ROOT_OUT = "/content/ghazal_tuning"
OUT = f"{ROOT_OUT}/{MODEL}-{SIZE}"
os.makedirs(OUT, exist_ok=True)
print("Outputs:", OUT)

In [ ]:
# Progress tracking: every stage is timed, logged to OUT/progress.jsonl and summarized at the end.
class Progress:
    def __init__(self, path):
        self.path, self.stages, self._t = path, [], {}
    def start(self, name, note=""):
        self._t[name] = time.time()
        print(f"\n▶ {name} {('· ' + note) if note else ''}  [{time.strftime('%H:%M:%S')}]")
    def done(self, name, **info):
        dt = time.time() - self._t.pop(name, time.time())
        rec = {"stage": name, "seconds": round(dt, 1), "at": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
        self.stages.append(rec)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        extra = " · ".join(f"{k}={v}" for k, v in info.items())
        print(f"✔ {name} in {fmt_dur(dt)}{(' · ' + extra) if extra else ''}")
    def table(self):
        return pd.DataFrame(self.stages)[["stage", "seconds"]].assign(minutes=lambda d: (d.seconds / 60).round(1))

def fmt_dur(s):
    s = int(s)
    return f"{s // 3600}h{(s % 3600) // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"

PROG = Progress(f"{OUT}/progress.jsonl")

## 3. Corpus

The public sample: ~1,300 ghazals by 30 poets, each in Rekhta's three scripts, line-aligned ([amir9ume/urdu_ghazals_rekhta](https://github.com/amir9ume/urdu_ghazals_rekhta), MIT; text from [Rekhta](https://www.rekhta.org)). Set `CORPUS_JSONL` to use your own export instead.

In [ ]:
SCRIPTS = ("nastaliq", "devanagari", "roman")
LABEL = {"nastaliq": "Nastaliq", "devanagari": "Devanagari", "roman": "Roman"}
FOLDERS = {"nastaliq": "ur", "devanagari": "hi", "roman": "en"}

POET_NAMES = {   # display names per script, for attribution options
    "mirza-ghalib": ("Mirza Ghalib", "مرزا غالب", "मिर्ज़ा ग़ालिब"), "meer-taqi-meer": ("Mir Taqi Mir", "میر تقی میر", "मीर तक़ी मीर"),
    "dagh-dehlvi": ("Dagh Dehlvi", "داغ دہلوی", "दाग़ देहलवी"), "bahadur-shah-zafar": ("Bahadur Shah Zafar", "بہادر شاہ ظفر", "बहादुर शाह ज़फ़र"),
    "allama-iqbal": ("Allama Iqbal", "علامہ اقبال", "अल्लामा इक़बाल"), "faiz-ahmad-faiz": ("Faiz Ahmad Faiz", "فیض احمد فیض", "फ़ैज़ अहमद फ़ैज़"),
    "ahmad-faraz": ("Ahmad Faraz", "احمد فراز", "अहमद फ़राज़"), "parveen-shakir": ("Parveen Shakir", "پروین شاکر", "परवीन शाकिर"),
    "jaun-eliya": ("Jaun Elia", "جون ایلیا", "जौन एलिया"), "firaq-gorakhpuri": ("Firaq Gorakhpuri", "فراق گورکھپوری", "फ़िराक़ गोरखपुरी"),
    "jigar-moradabadi": ("Jigar Moradabadi", "جگر مرادآبادی", "जिगर मुरादाबादी"), "akbar-allahabadi": ("Akbar Allahabadi", "اکبر الہ آبادی", "अकबर इलाहाबादी"),
    "habib-jalib": ("Habib Jalib", "حبیب جالب", "हबीब जालिब"), "sahir-ludhianvi": ("Sahir Ludhianvi", "ساحر لدھیانوی", "साहिर लुधियानवी"),
    "nida-fazli": ("Nida Fazli", "ندا فاضلی", "निदा फ़ाज़ली"), "waseem-barelvi": ("Waseem Barelvi", "وسیم بریلوی", "वसीम बरेलवी"),
    "javed-akhtar": ("Javed Akhtar", "جاوید اختر", "जावेद अख़्तर"), "wali-mohammad-wali": ("Wali Mohammad Wali", "ولی محمد ولی", "वली मोहम्मद वली"),
    "mohsin-naqvi": ("Mohsin Naqvi", "محسن نقوی", "मोहसिन नक़वी"), "gulzar": ("Gulzar", "گلزار", "गुलज़ार"),
    "altaf-hussain-hali": ("Altaf Hussain Hali", "الطاف حسین حالی", "अल्ताफ़ हुसैन हाली"),
    "jaan-nisar-akhtar": ("Jaan Nisar Akhtar", "جاں نثار اختر", "जाँ निसार अख़्तर"),
}
def poet_name(slug, lang):
    n = POET_NAMES.get(slug)
    fallback = slug.replace("-", " ").title()
    if not n: return fallback
    return {"en": n[0], "roman": n[0], "nastaliq": n[1], "devanagari": n[2]}.get(lang, n[0])

def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [l.strip() for l in f.read().splitlines() if l.strip()]

def load_rekhta_tree(root):
    poems = []
    for poet in sorted(os.listdir(root)):
        pdir = os.path.join(root, poet)
        if not os.path.isdir(os.path.join(pdir, "en")):
            continue
        files = set()
        for d in FOLDERS.values():
            if os.path.isdir(os.path.join(pdir, d)):
                files |= {f for f in os.listdir(os.path.join(pdir, d)) if not f.startswith(".")}
        for fname in sorted(files):
            text = {s: read_lines(os.path.join(pdir, d, fname)) for s, d in FOLDERS.items()
                    if os.path.isfile(os.path.join(pdir, d, fname))}
            text = {s: v for s, v in text.items() if v}
            if text:
                poems.append({"id": f"{poet}/{fname}", "poet": poet, "form": "ghazal", "text": text})
    return poems

PROG.start("corpus")
if CORPUS_JSONL:
    with open(CORPUS_JSONL, encoding="utf-8") as f:
        poems_all = [json.loads(l) for l in f if l.strip()]
    for p in poems_all:   # poet display names from the export, if it has them
        n = p.get("poet_names") or {}
        if n and p["poet"] not in POET_NAMES:
            en = n.get("en") or p["poet"].replace("-", " ").title()
            POET_NAMES[p["poet"]] = (en, n.get("nastaliq", en), n.get("devanagari", en))
else:
    if not os.path.exists("/content/rekhta_sample/dataset"):
        if not os.path.exists("/content/urdu_ghazals_rekhta"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/amir9ume/urdu_ghazals_rekhta.git",
                            "/content/urdu_ghazals_rekhta"], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        zipfile.ZipFile("/content/urdu_ghazals_rekhta/dataset/dataset.zip").extractall("/content/rekhta_sample")
    poems_all = load_rekhta_tree("/content/rekhta_sample/dataset")

def aligned(p):
    lens = {len(v) for v in p["text"].values()}
    return len(lens) == 1 and min(lens) >= 2
poems = [p for p in poems_all if aligned(p) and p.get("form", "ghazal") == "ghazal"]
PROG.done("corpus", poems=len(poems), poets=len({p["poet"] for p in poems}), skipped=len(poems_all) - len(poems))

## 4. Text utilities and rule-based form checks

Radif and qafiya are extracted by rule, not learned. Here they serve as checks: a correct transliteration of a matla keeps its radif and qafiya. A radif found in only some scripts (Urdu writes the suffix گے as a separate word; Hindi joins it) isn't trusted.

In [ ]:
_MARKS = re.compile("[ً-ٰٟँ़]")
_ROMAN_JOIN = re.compile(r"[.'’ʼ‘]")
_IZAFAT = re.compile(r"-(?:e|o|ए|ओ)-")

def tokens(text, script):
    # Loose tokens: diacritics, nukta, macrons and punctuation removed (for rule checks and loose CER).
    s = unicodedata.normalize("NFC", str(text).lower())
    if script == "roman":
        s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
        s = _ROMAN_JOIN.sub("", s)
    s = _MARKS.sub("", s)
    s = "".join(" " if unicodedata.category(c)[0] in "PSZ" else c for c in s)
    return s.split()

def loose(text, script):
    return " ".join(tokens(text, script))

def strict(text):
    # Strict form: Unicode-normalized, whitespace collapsed, typographic quotes unified. Case is kept:
    # Rekhta's Roman uses capitals for retroflex consonants (DhūñDh, ujḌe).
    s = unicodedata.normalize("NFC", str(text)).replace("’", "'").replace("‘", "'")
    return "\n".join(" ".join(l.split()) for l in s.strip().splitlines() if l.strip())

def detect_script(text):
    c = {"nastaliq": sum("؀" <= ch <= "ۿ" for ch in text),
         "devanagari": sum("ऀ" <= ch <= "ॿ" for ch in text),
         "roman": sum(ch.isalpha() and unicodedata.normalize("NFKD", ch)[0].isascii() for ch in text)}
    best = max(c, key=c.get)
    return best if c[best] else None

def radif(l1, l2, script):
    w1, w2, shared = l1.split(), l2.split(), []
    for a, b in zip(reversed(w1), reversed(w2)):
        if not loose(a, script) or loose(a, script) != loose(b, script):
            break
        shared.insert(0, b)
    return " ".join(shared)

def qafiya_words(l1, l2, script):
    r = radif(l1, l2, script)
    if not r:
        return None
    n, w1, w2 = len(r.split()), l1.split(), l2.split()
    if len(w1) <= n or len(w2) <= n:
        return None
    q1, q2 = (_IZAFAT.split(w[len(w) - n - 1])[-1] for w in (w1, w2))
    if not loose(q1, script) or not loose(q2, script) or loose(q1, script) == loose(q2, script):
        return None
    return q1, q2

def strip_think(text):
    if "</think>" in text: return text.split("</think>")[-1].strip(), False
    if "<think>" in text: return "", True
    return text.strip(), False

def shers(p, script):
    L = p["text"][script]
    return [(L[i], L[i + 1]) for i in range(0, len(L) - 1, 2)]

def scripts_of(p):
    return [s for s in SCRIPTS if p["text"].get(s)]

def edit_distance(a, b):
    # Levenshtein distance over sequences (characters or words).
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

g = next(p for p in poems if p["poet"] == "mirza-ghalib")
for s in scripts_of(g):
    print(f"{LABEL[s]:<11} radif: {radif(*g['text'][s][:2], s)!r:<14} qafiya: {qafiya_words(*g['text'][s][:2], s)}")

## 5. The task: transliterate a sher

Six directions. Training mixes directions and the language of the instruction (English, Urdu, Hindustani in Devanagari, Roman Urdu); evaluation uses English instructions so only the scripts vary.

In [ ]:
SCRIPT_NAMES = {
    "en": {"nastaliq": "Urdu script (Nastaliq)", "devanagari": "Devanagari", "roman": "Roman script (Latin letters)"},
    "nastaliq": {"nastaliq": "اردو رسم الخط", "devanagari": "دیوناگری", "roman": "رومن رسم الخط"},
    "devanagari": {"nastaliq": "उर्दू लिपि", "devanagari": "देवनागरी", "roman": "रोमन लिपि"},
    "roman": {"nastaliq": "Urdu script", "devanagari": "Devanagari", "roman": "Roman"},
}
TEMPLATE = {
    "en": "Transliterate this sher into {script}. Keep every word as it is; write only the transliterated sher, in two lines.",
    "nastaliq": "اس شعر کو {script} میں لکھیے۔ ہر لفظ برقرار رکھیے؛ صرف شعر، دو مصرعوں میں، لکھیے۔",
    "devanagari": "इस शेर को {script} में लिखिए। हर लफ़्ज़ बरक़रार रखिए; सिर्फ़ शेर, दो मिसरों में, लिखिए।",
    "roman": "Is sher ko {script} mein likhiye. Har lafz barqaraar rakhiye; sirf sher, do misron mein, likhiye.",
}
DIRECTIONS = [(a, b) for a in SCRIPTS for b in SCRIPTS if a != b]
DIR_LABEL = {d: f"{LABEL[d[0]]} → {LABEL[d[1]]}" for d in DIRECTIONS}

def make_example(p, src, tgt, idx, lang="en"):
    sc = scripts_of(p)
    if src not in sc or tgt not in sc: return None
    a, b = shers(p, src), shers(p, tgt)
    if idx >= min(len(a), len(b)): return None
    prompt = TEMPLATE[lang].format(script=SCRIPT_NAMES[lang][tgt]) + f"\n\n{a[idx][0]}\n{a[idx][1]}"
    e = {"id": f"{p['id']}|s{idx}|{src}>{tgt}|{lang}", "poem_id": p["id"], "poet": p["poet"], "sher": idx,
         "src": src, "tgt": tgt, "lang": lang, "prompt": prompt, "answer": f"{b[idx][0]}\n{b[idx][1]}",
         "source": list(a[idx]), "item_key": f"{p['id']}|s{idx}"}
    if idx == 0:   # matla: record rule-based form, if all scripts agree on a radif
        rads = {s: radif(*p["text"][s][:2], s) for s in sc}
        if all(rads.values()):
            e["form"] = {"radif": rads[tgt], "qafiya": qafiya_words(*p["text"][tgt][:2], tgt)}
    return e

ex = make_example(g, "nastaliq", "devanagari", 0, "roman")
print(ex["prompt"], "\n-->\n" + ex["answer"], "\nform:", ex.get("form"))

## 6. Train / validation / test

Splits are by poem (a hash of the poem id, so they don't shift when data is added). Two poets are held out entirely.

* **train**: the only data the model learns from
* **validation**: scored during training to track progress and pick the best checkpoint; never reported as a result
* **test_seen**: new poems by poets seen in training · **test_unseen**: poets never seen in training

Each test poem contributes its matla and one middle sher, in all six directions.

In [ ]:
HOLDOUT_POETS = ["jigar-moradabadi", "waseem-barelvi"]
TEST_FRAC, VAL_FRAC = 0.15, 0.05
LANG_WEIGHTS = {"en": 0.4, "nastaliq": 0.2, "devanagari": 0.2, "roman": 0.2}

def unit(key):
    return int(hashlib.sha1(f"{SEED}:{key}".encode()).hexdigest()[:12], 16) / float(0xFFFFFFFFFFFF)

split_of = {}
for p in poems:
    u = unit(p["id"])
    split_of[p["id"]] = ("test_unseen" if p["poet"] in HOLDOUT_POETS else
                         "test_seen" if u < TEST_FRAC else "val" if u < TEST_FRAC + VAL_FRAC else "train")
by_split = defaultdict(list)
for p in poems:
    by_split[split_of[p["id"]]].append(p)

def eval_shers(p):
    n = len(shers(p, scripts_of(p)[0]))
    return sorted({0, n // 2 if n >= 3 else n - 1})

def eval_set(ps):
    return [e for p in ps for i in eval_shers(p) for (s, t) in DIRECTIONS if (e := make_example(p, s, t, i))]

def limit_poems(ps, n):
    return sorted(ps, key=lambda p: unit("pick:" + p["id"]))[:n] if n else ps

PROG.start("datasets")
r = random.Random(SEED)
pool = []                                   # every (poem, sher, direction) in train, with a random instruction language
for p in by_split["train"]:
    for i in range(len(shers(p, scripts_of(p)[0]))):
        for s, t in DIRECTIONS:
            lang = r.choices(list(LANG_WEIGHTS), weights=list(LANG_WEIGHTS.values()))[0]
            if (e := make_example(p, s, t, i, lang)):
                pool.append(e)
r.shuffle(pool)
TRAIN = pool[: P["n_train"]] if P["n_train"] else pool
TEST = {s: eval_set(limit_poems(by_split[s], P["eval_poems"])) for s in ("test_seen", "test_unseen")}
PROBE = [e for p in limit_poems(by_split["val"], P["probe_poems"]) for (s, t) in DIRECTIONS if (e := make_example(p, s, t, 0))]
PROG.done("datasets", train=len(TRAIN), train_available=len(pool), test_seen=len(TEST["test_seen"]),
          test_unseen=len(TEST["test_unseen"]), validation_probe=len(PROBE))

display(pd.DataFrame({
    "poems": {s: len(by_split[s]) for s in ("train", "val", "test_seen", "test_unseen")},
    "poets": {s: len({p["poet"] for p in by_split[s]}) for s in ("train", "val", "test_seen", "test_unseen")},
    "used poems": {"train": len({e["poem_id"] for e in TRAIN}), "val": len({e["poem_id"] for e in PROBE}),
                   "test_seen": len({e["poem_id"] for e in TEST["test_seen"]}),
                   "test_unseen": len({e["poem_id"] for e in TEST["test_unseen"]})},
}))
print("Training directions:", {DIR_LABEL[k]: v for k, v in Counter((e["src"], e["tgt"]) for e in TRAIN).items()})
print("Instruction languages:", dict(Counter(e["lang"] for e in TRAIN)))
assert not ({e["poem_id"] for e in TRAIN} & {e["poem_id"] for v in TEST.values() for e in v}), "train/test overlap!"

## 7. Metrics

* **CER / WER (strict):** character and word error rate against Rekhta's text, every nukta, macron and vowel sign counted. Lower is better.
* **CER (loose):** the same with diacritics, nukta and macrons removed. *Strict minus loose* = errors that are only orthographic detail.
* **Exact:** share of shers transliterated perfectly (strict).
* **Phenomena:** word accuracy on words containing a given feature in the target script.
* **Form kept:** for a matla, does the transliteration still have the right radif and qafiya (rule-based)?

Answers are extracted robustly: reasoning blocks, labels and quotes are stripped; lines that just repeat the source are dropped.

In [ ]:
_LABEL = re.compile(r"^\s*(?:\*\*)?(transliteration|answer|here is|output|ترجمہ|लिप्यंतरण)[^:：]*[:：]\s*", re.I)

def extract(raw, e):
    ans, unfinished = strip_think(raw)
    if unfinished:
        return None
    lines, echoed = [], []
    for l in ans.splitlines():
        l = _LABEL.sub("", l.strip()).strip(' "“”«»*`').strip()
        if not l:
            continue
        if detect_script(l) == e["src"] != e["tgt"] and any(
                SequenceMatcher(None, loose(l, e["src"]), loose(x, e["src"])).ratio() > 0.85 for x in e["source"]):
            echoed.append(l)                                  # repeats the source verse
            continue
        lines.append(l)
    want = [l for l in lines if detect_script(l) == e["tgt"]]
    lines = want or lines or echoed       # only echoed the source: score that (it's a wrong-script answer)
    if len(lines) == 1 and " / " in lines[0]:
        lines = lines[0].split(" / ")
    return "\n".join(lines[:2])

PHENOMENA = {
    "devanagari": {
        "nukta (ज़ ख़ क़ फ़ ग़)": lambda w: "़" in w,
        "aspirates (ख घ थ ध फ भ …)": lambda w: any(c in "खघछझठढथधफभ" for c in w),
        "nasalization (ँ ं)": lambda w: "ँ" in w or "ं" in w,
        "conjuncts (्)": lambda w: "्" in w,
        "izafat (-ए- / -ओ-)": lambda w: bool(re.search("-(ए|ओ)-", w)),
    },
    "roman": {
        "long vowels (ā ī ū)": lambda w: any(c in w for c in "āīū"),
        "nasalization (ñ)": lambda w: "ñ" in w,
        "aspirates (kh gh th dh ph bh …)": lambda w: bool(re.search(r"(?<!ḳ)[kgcjtdpbTD]h", w)),
        "retroflex (T D R Ḍ Ṛ)": lambda w: any(c in w for c in "TDRḌṚṬ"),
        "Perso-Arabic sounds (ḳh ġ q z f)": lambda w: bool(re.search("ḳh|ġ|q|z|f", w)),
        "izafat (-e- / -o-)": lambda w: bool(re.search("-(e|o)-", w)),
    },
    "nastaliq": {
        "aspirates (ھ)": lambda w: "ھ" in w,
        "noon ghunna (ں)": lambda w: "ں" in w,
        "hamza / izafat (ء ئ ۂ ؤ)": lambda w: any(c in w for c in "ءئۂؤ"),
        "spelling choice (ث ذ ص ض ط ظ ع ح)": lambda w: any(c in w for c in "ثذصضطظعح"),
        "bari ye (ے)": lambda w: "ے" in w,
    },
}

def word_hits(gold, pred):
    # Which gold words appear correctly (strict) in the prediction, by word alignment.
    gw, pw = strict(gold).split(), strict(pred).split()
    ok = [False] * len(gw)
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, gw, pw, autojunk=False).get_opcodes():
        if tag == "equal":
            for i in range(i1, i2): ok[i] = True
    return gw, ok

def score(e, raw):
    pred = extract(raw, e)
    gold = e["answer"]
    r = {"no_answer": pred is None or not pred.strip()}
    pred = pred or ""
    gs, ps = strict(gold), strict(pred)
    gl, pl = loose(gold, e["tgt"]), loose(pred, e["tgt"])
    r["pred"] = pred
    r["chars"], r["char_err"] = len(gs), min(edit_distance(gs, ps), len(gs))
    r["chars_loose"], r["char_err_loose"] = len(gl), min(edit_distance(gl, pl), len(gl))
    r["words"], r["word_err"] = len(gs.split()), min(edit_distance(gs.split(), ps.split()), len(gs.split()))
    r["exact"] = gs == ps
    r["wrong_script"] = bool(pred.strip()) and detect_script(pred) != e["tgt"]
    gw, ok = word_hits(gold, pred)
    r["phen"] = {name: [sum(o for w, o in zip(gw, ok) if f(w)), sum(1 for w in gw if f(w))]
                 for name, f in PHENOMENA[e["tgt"]].items()}
    if "form" in e:
        pl2 = (pred.splitlines() + ["", ""])[:2]
        rr = radif(pl2[0], pl2[1], e["tgt"])
        r["radif_kept"] = loose(rr, e["tgt"]) == loose(e["form"]["radif"], e["tgt"]) and bool(rr)
        if e["form"]["qafiya"]:
            q = qafiya_words(pl2[0], pl2[1], e["tgt"])
            r["qafiya_kept"] = bool(q) and [loose(x, e["tgt"]) for x in q] == [loose(x, e["tgt"]) for x in e["form"]["qafiya"]]
    return r

def summarize(rows, by=("split", "src", "tgt")):
    df = pd.DataFrame(rows)
    g = df.groupby(list(by))
    out = pd.DataFrame({
        "CER": g.char_err.sum() / g.chars.sum(),
        "CER loose": g.char_err_loose.sum() / g.chars_loose.sum(),
        "WER": g.word_err.sum() / g.words.sum(),
        "exact": g.exact.mean(),
        "no answer": g.no_answer.mean(),
        "wrong script": g.wrong_script.mean(),
        "n": g.size(),
    })
    return out

def phenomena_table(rows, split):
    acc = defaultdict(lambda: [0, 0])
    for r in rows:
        if r["split"] == split:
            for name, (hit, tot) in r["phen"].items():
                acc[(LABEL[r["tgt"]], name)][0] += hit
                acc[(LABEL[r["tgt"]], name)][1] += tot
    return pd.Series({k: h / t for k, (h, t) in acc.items() if t}, name="word accuracy")

def form_rates(rows, split):
    d = pd.DataFrame([r for r in rows if r["split"] == split])
    out = {}
    for k in ("radif_kept", "qafiya_kept"):
        if k in d:
            v = d[k].dropna()
            out[k] = float(v.mean()) if len(v) else float("nan")
    return out

# sanity checks on known cases
e = make_example(g, "nastaliq", "devanagari", 0)
perfect, nukta_only = score(e, e["answer"]), score(e, e["answer"].replace("़", ""))
assert perfect["exact"] and perfect["char_err"] == 0 and perfect.get("radif_kept")
assert nukta_only["char_err"] > 0 and nukta_only["char_err_loose"] == 0     # orthography-only error
assert score(e, "\n".join(e["source"]))["char_err"] > 0                      # echo is not an answer
print("metrics OK · e.g. nukta-only errors:", nukta_only["char_err"], "chars strict, 0 loose")

## 8. Load the model (once)

The model is loaded once and reused for the baseline, training and the tuned evaluation. For Sarvam-30B the first load downloads ~129 GB.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PROG.start("load model", f"{M['id']} ({QUANT})")
tok = AutoTokenizer.from_pretrained(M["id"], trust_remote_code=M["remote_code"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
load_kw = {"device_map": "auto", "trust_remote_code": M["remote_code"], "torch_dtype": torch.bfloat16}
if QUANT == "4bit":
    load_kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16, llm_int8_skip_modules=["lm_head"])
try:
    model = AutoModelForCausalLM.from_pretrained(M["id"], **load_kw)
except ImportError as err:
    raise RuntimeError(f"{err}\n\nThe model's custom code may not match this transformers version. "
                       f"Set TRANSFORMERS_VERSION = '<5' in Settings, then Runtime > Restart and Run all.") from err
model.eval()
PROG.done("load model", gpu_gb=round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
# Prompt rendering and batched generation, shared by evaluation and training (so formats can't drift).
THINK_PREFILL = False   # decided in the preflight cell
GEN_USE_CACHE = True    # the sanity gate may turn this off if cached generation is broken

def render(prompt_text):
    p = tok.apply_chat_template([{"role": "user", "content": prompt_text}], tokenize=False,
                                add_generation_prompt=True, **M["chat_kwargs"])
    if THINK_PREFILL:
        tail = p.rstrip()
        if tail.endswith("<think>"):
            p = p + "\n\n</think>\n\n"
        elif "</think>" not in p[-60:]:
            p = p + "<think>\n\n</think>\n\n"
    return p

_SPECIALS = None
def clean_specials(text):
    global _SPECIALS
    if _SPECIALS is None:   # keep <think> tags (the scorer needs them), drop pad/eos/etc.
        _SPECIALS = [t for t in set(tok.all_special_tokens) if t and "think" not in t.lower()]
    for t in _SPECIALS:
        text = text.replace(t, "")
    return text.strip()

@torch.no_grad()
def generate(prompts, max_new_tokens, batch_size=None, desc="generating", show=True):
    batch_size = batch_size or M["gen_batch"]
    texts = [render(p) for p in prompts]
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))   # longest first: OOM shows up early
    outs = [""] * len(texts)
    bar = tqdm(total=len(texts), desc=desc, disable=not show, leave=False)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False,
                  return_token_type_ids=False).to(model.device)
        enc = {k: enc[k] for k in ("input_ids", "attention_mask") if k in enc}   # Sarvam's tokenizer adds token_type_ids, which generate() rejects
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=GEN_USE_CACHE,
                             pad_token_id=tok.pad_token_id)
        for i, t in zip(idx, tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)):
            outs[i] = clean_specials(t)
        bar.update(len(idx))
    bar.close()
    return outs

## 9. Preflight

1. **Tokenizer coverage** per script: tokens per word and exact round-trip.
2. **Reasoning behavior:** does the model think before answering? (`THINK_MODE="auto"` pre-fills an empty, closed think block if so, for base and tuned alike.)
3. **Speed:** a timed batch, to estimate each evaluation's duration.

In [ ]:
PROG.start("preflight")
rows = []
sample = [p for p in poems if len(scripts_of(p)) == 3][:150]
for sc in SCRIPTS:
    lines = [l for p in sample for l in p["text"][sc]]
    n_tok = n_words = rt = 0
    for l in lines:
        ids = tok(l, add_special_tokens=False)["input_ids"]
        n_tok += len(ids); n_words += len(l.split())
        rt += tok.decode(ids, skip_special_tokens=True).strip() == l.strip()
    rows.append({"script": LABEL[sc], "tokens/word": round(n_tok / n_words, 2), "round-trip": f"{rt / len(lines):.1%}"})
display(pd.DataFrame(rows).set_index("script"))

# ---- 1. Does the model reason before answering? Decide this first: a reasoning model can spend
#         the whole token budget thinking, which would look like "no answer" to every later check.
probe_ex = make_example(g, "nastaliq", "devanagari", 0)
_REASONING = re.compile(r"^\s*(<think>|\d+\.\s+\*\*Analy[sz]e|\*\*Analy[sz]e|Okay, |Let me |First, )", re.I)
def reasons(text): return "<think>" in text or "</think>" in text or bool(_REASONING.match(text))

def detect_reasoning():
    global THINK_PREFILL, EXTRA
    THINK_PREFILL, EXTRA = False, 0
    raw_free = generate([probe_ex["prompt"]], 256, show=False)[0]
    thinks = reasons(raw_free) or "<think>" in render(probe_ex["prompt"])[-40:]
    print("Raw output without prefill:\n", raw_free[:400], "\n")
    if THINK_MODE == "prefill" or (THINK_MODE == "auto" and thinks):
        THINK_PREFILL = True
        raw_pre = generate([probe_ex["prompt"]], ANSWER_TOKENS, show=False)[0]
        if reasons(raw_pre):
            # The model ignores an empty think block and reasons anyway: let it think, then keep only the answer.
            print("⚠️  The model keeps reasoning even with an empty think block pre-filled → allowing reasoning "
                  f"with {THINK_TOKENS} extra tokens per answer; only the text after </think> is scored.")
            THINK_PREFILL, EXTRA = False, THINK_TOKENS
    elif THINK_MODE == "allow" and thinks:
        EXTRA = THINK_TOKENS
    print(f"Model reasons before answering: {thinks} → think prefill {'ON' if THINK_PREFILL else 'OFF'}"
          + (f", reasoning allowed (+{EXTRA} tokens)" if EXTRA else ""))
    return thinks

thinks = detect_reasoning()

# ---- 2. Sanity gate: is the model producing language at all? (Stops here instead of after an hour of baseline.)
def sanity():
    budget = 512 + EXTRA
    a = generate(["What is the capital of France? Answer with one word."], budget, show=False)[0]
    pe = make_example(g, "roman", "devanagari", 0)       # the easiest direction: nearly mechanical
    b = generate([pe["prompt"]], budget, show=False)[0]
    ans, unfinished = strip_think(b)
    if unfinished or (not ans and reasons(b)):
        ans = ""
    words = ans.split()
    letters = [c for c in ans if c.isalpha() or "ऀ" <= c <= "ॿ"]
    deva = sum("ऀ" <= c <= "ॿ" for c in ans)
    checks = {"knows 'Paris'": "paris" in a.lower(),
              "writes Devanagari": bool(letters) and deva / len(letters) > 0.3,   # commentary alongside is fine
              "not degenerate repetition": len(words) > 3 and len(set(words)) / len(words) > 0.3}
    return all(checks.values()), checks, a, b

ok, checks, a, b = sanity()
print("Sanity:", checks)
if not ok and reasons(b) and not EXTRA:
    print("The model is reasoning, not broken: it ran out of tokens before answering. "
          f"Allowing reasoning (+{THINK_TOKENS} tokens) and retrying...")
    THINK_PREFILL, EXTRA = False, THINK_TOKENS
    ok, checks, a, b = sanity()
    print("Sanity with reasoning allowed:", checks)
if not ok and not reasons(b):
    print("Output looks broken. Sample:", repr(a[:200]), "|", repr(b[:200]))
    print("Retrying without the KV cache (a common incompatibility in custom model code)...")
    GEN_USE_CACHE = False
    ok, checks, a, b = sanity()
    print("Sanity without cache:", checks)
    if ok:
        print("✅ Works without the KV cache: generation will be slower but correct. Continuing.")
        thinks = detect_reasoning()          # the first check ran on broken output; redo it
        ok, checks, a, b = sanity()
if not ok:
    raise RuntimeError(
        ("The model reasons at length and never reached an answer, even with extra tokens. Try THINK_TOKENS = 2048.\n"
         if reasons(b) else "The model's output is degenerate, so the baseline would be meaningless. Stopping before it.\n")
        + f"Environment: transformers {transformers.__version__}, torch {torch.__version__}, quantization {QUANT}.\n"
        f"Sample output: {b[-300:]!r}\n"
        "Fixes to try: (1) Settings → TRANSFORMERS_VERSION = '==4.57.1' (or another 4.x), then Runtime → "
        "Disconnect and delete runtime → Run all; (2) MODEL = 'qwen2.5-7b' to validate the pipeline meanwhile.")
print("✅ Model output is sane:", repr(strip_think(b)[0][:120]))

raw_used = generate([probe_ex["prompt"]], ANSWER_TOKENS + EXTRA, show=False)[0]
print("Prompt tail the model sees:", repr(render(probe_ex["prompt"])[-100:]))
print("Gold:\n" + probe_ex["answer"])
s = score(probe_ex, raw_used)
print(f"Model:\n{s['pred']}\n→ CER {s['char_err'] / max(s['chars'], 1):.1%}")

t0 = time.time()
_ = generate([e["prompt"] for e in TEST["test_seen"][:M["gen_batch"]]], ANSWER_TOKENS + EXTRA, show=False)
SEC_PER_BATCH = time.time() - t0
n_eval = sum(len(v) for v in TEST.values())
est = SEC_PER_BATCH * math.ceil(n_eval / M["gen_batch"]) * 1.1
print(f"\nOne batch of {M['gen_batch']}: {SEC_PER_BATCH:.1f}s → each evaluation ({n_eval} prompts) ≈ {fmt_dur(est)}")
PROG.done("preflight", think_prefill=THINK_PREFILL, est_eval_min=round(est / 60, 1))

## 10. Evaluation runner

Predictions are appended to disk as they finish; re-running after a disconnect resumes instead of starting over.

In [ ]:
def run_eval(tag, splits, show=True):
    path = f"{OUT}/predictions_{tag}.jsonl"
    done = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for l in f:
                r = json.loads(l); done[(r["split"], r["id"])] = r
    todo = [(s, e) for s, exs in splits.items() for e in exs if (s, e["id"]) not in done]
    if done and show:
        print(f"Resuming {tag}: {len(done)} done, {len(todo)} to go")
    with open(path, "a", encoding="utf-8") as f:
        for c in range(0, len(todo), 128):                 # flush to disk every 128 prompts
            chunk = todo[c:c + 128]
            raws = generate([e["prompt"] for _, e in chunk], ANSWER_TOKENS + EXTRA,
                            desc=f"{tag} {c + len(chunk)}/{len(todo)}", show=show)
            for (s, e), raw in zip(chunk, raws):
                row = {"split": s, "id": e["id"], "poet": e["poet"], "src": e["src"], "tgt": e["tgt"],
                       "sher": e["sher"], "gold": e["answer"], "raw": raw, **score(e, raw)}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                done[(s, e["id"])] = row
            f.flush()
    return [done[(s, e["id"])] for s, exs in splits.items() for e in exs if (s, e["id"]) in done]

def direction_table(rows, split):
    t = summarize([r for r in rows if r["split"] == split]).droplevel(0)
    t.index = [DIR_LABEL[i] for i in t.index]
    t.loc["all directions"] = summarize([dict(r, split="x", src="all", tgt="all") for r in rows if r["split"] == split]).iloc[0]
    return t

## 11. Before tuning: baseline on the test sets

In [ ]:
PROG.start("baseline eval", f"{n_eval} prompts")
base_rows = run_eval("base", TEST)
PROG.done("baseline eval", no_answer=f"{np.mean([r['no_answer'] for r in base_rows]):.1%}")
fmt = {"CER": "{:.1%}", "CER loose": "{:.1%}", "WER": "{:.1%}", "exact": "{:.1%}", "no answer": "{:.1%}", "wrong script": "{:.1%}", "n": "{:.0f}"}
for split in TEST:
    print(f"\n{split} (lower CER/WER is better)")
    display(direction_table(base_rows, split).style.format(fmt).background_gradient(subset=["CER"], cmap="Reds", vmin=0, vmax=0.6))

## 12. LoRA fine-tuning, validated as it goes

* Loss on the answer tokens only; prompts rendered exactly as at evaluation.
* Every `PROBE_EVERY` steps the **validation** probe (matla of held-out validation poems, six directions) is scored. The adapter with the **lowest validation CER** is kept, and that is the checkpoint evaluated on test.
* The validation curve, error against examples seen, shows what more data buys.

In [ ]:
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from transformers import Trainer, TrainingArguments, TrainerCallback

def featurize(exs):
    feats, dropped = [], 0
    for e in exs:
        p_ids = tok(render(e["prompt"]), add_special_tokens=False)["input_ids"]
        a_ids = tok(e["answer"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]
        if len(p_ids) + len(a_ids) > MAX_SEQ_LEN:
            dropped += 1; continue
        feats.append({"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids})
    return feats, dropped

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda x, v: x + [v] * (n - len(x))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch])}

train_feats, dropped = featurize(TRAIN)
print(f"{len(train_feats)} training sequences ({dropped} over {MAX_SEQ_LEN} tokens dropped)")

# LoRA on attention + shared/dense MLP; never the routed experts or the router
ATTN = ("q_proj", "k_proj", "v_proj", "o_proj", "query_key_value", "qkv_proj", "dense", "out_proj")
MLP = ("gate_proj", "up_proj", "down_proj")
targets = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear) and
           ((n.rsplit(".", 1)[-1] in ATTN and "expert" not in n) or (n.rsplit(".", 1)[-1] in MLP and ".experts." not in n))]
print(f"LoRA on {len(targets)} modules, e.g. {targets[:3]}")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05,
                                         target_modules=targets, bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

eff_batch = M["train_batch"] * M["grad_accum"]
total_steps = math.ceil(len(train_feats) / eff_batch) * P["epochs"]
PROBE_EVERY = max(10, total_steps // 8)            # ~8 validation points along the curve
print(f"{total_steps} optimizer steps · effective batch {eff_batch} · validation every {PROBE_EVERY} steps")

def probe_score():
    model.eval()
    raws = generate([e["prompt"] for e in PROBE], ANSWER_TOKENS + EXTRA, show=False)
    model.train()
    rows = [dict(score(e, r), src=e["src"], tgt=e["tgt"], split="val") for e, r in zip(PROBE, raws)]
    by_src = summarize(rows, by=("src",))["CER"].to_dict()
    total = summarize(rows, by=("split",)).iloc[0]
    return float(total["CER"]), float(total["exact"]), by_src

BEST_PATH = f"{OUT}/best_adapter_state.pt"      # best validation weights, kept on Drive for resume
CKPT_DIR = f"{OUT}/trainer"

class ValidationCurve(TrainerCallback):
    def __init__(self, resume_step=0):
        self.curve, self.best, self.best_state = [], float("inf"), None
        self.t0, self.start_step = time.time(), resume_step
        if resume_step and os.path.exists(f"{OUT}/validation_curve.csv"):     # pick up the curve from before the disconnect
            self.curve = [r for r in pd.read_csv(f"{OUT}/validation_curve.csv").to_dict("records") if r["step"] <= resume_step]
            if self.curve: self.best = min(r["val_CER"] for r in self.curve)
            if os.path.exists(BEST_PATH): self.best_state = torch.load(BEST_PATH)
            print(f"Resumed validation curve: {len(self.curve)} points, best CER so far {self.best:.1%}")
    def _probe(self, step):
        cer, exact, by_src = probe_score()
        seen = step * eff_batch
        self.curve.append({"step": step, "examples_seen": seen, "val_CER": cer, "val_exact": exact,
                           **{f"from {LABEL[k]}": v for k, v in by_src.items()}})
        mark = ""
        if cer < self.best:
            self.best, mark = cer, "★ best"
            self.best_state = {k: v.detach().to("cpu", copy=True) for k, v in get_peft_model_state_dict(model).items()}
            torch.save(self.best_state, BEST_PATH)
        elapsed, done = time.time() - self.t0, step - self.start_step
        eta = f" · ETA {fmt_dur(elapsed / done * (total_steps - step))}" if done > 0 else ""
        src = " ".join(f"{LABEL[k][:4]}={v:.1%}" for k, v in by_src.items())
        print(f"[val] step {step}/{total_steps} · seen {seen:,} · CER {cer:.1%} · exact {exact:.0%} · from {src} {mark}{eta}")
        pd.DataFrame(self.curve).to_csv(f"{OUT}/validation_curve.csv", index=False)
    def on_train_begin(self, args, state, control, **kw):
        if not self.start_step: self._probe(0)
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % PROBE_EVERY == 0:
            self._probe(state.global_step)
    def on_train_end(self, args, state, control, **kw):
        if not self.curve or self.curve[-1]["step"] != state.global_step:
            self._probe(state.global_step)
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            print(f"   step {state.global_step}/{total_steps} · loss {logs['loss']:.3f} · lr {logs.get('learning_rate', 0):.1e}")

import inspect
targs = dict(output_dir=f"{OUT}/trainer", num_train_epochs=P["epochs"],
             per_device_train_batch_size=M["train_batch"], gradient_accumulation_steps=M["grad_accum"],
             learning_rate=LEARNING_RATE, warmup_steps=max(1, round(0.03 * total_steps)), lr_scheduler_type="cosine",
             logging_steps=max(1, PROBE_EVERY // 4), report_to="none",
             save_strategy="steps", save_steps=PROBE_EVERY, save_total_limit=1,   # checkpoint to Drive for resume
             bf16=True, remove_unused_columns=False, seed=SEED, disable_tqdm=False)
accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
skipped = [k for k in targs if k not in accepted]
targs = {k: v for k, v in targs.items() if k in accepted}
if skipped:
    print("TrainingArguments: this transformers version doesn't take", skipped, "(skipped)")

DONE_PATH = f"{OUT}/training_done.json"
import glob
if os.path.exists(DONE_PATH) and os.path.exists(f"{OUT}/adapter"):
    # Training already finished in an earlier session: load its adapter and logs instead of retraining.
    from peft.utils import load_peft_weights
    set_peft_model_state_dict(model, load_peft_weights(f"{OUT}/adapter"))
    LOSS_LOG = json.load(open(DONE_PATH))["log_history"]
    val_cb = ValidationCurve(resume_step=10**9)
    print(f"✅ Training already done in an earlier session: loaded the saved adapter from {OUT}/adapter")
else:
    ckpts = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1]))
    resume = ckpts[-1] if ckpts else None
    resume_step = int(resume.rsplit("-", 1)[-1]) if resume else 0
    if resume:
        print(f"↻ Resuming training from {resume} (step {resume_step}/{total_steps}) after a disconnect")
    val_cb = ValidationCurve(resume_step)
    trainer = Trainer(
        model=model, train_dataset=train_feats, data_collator=collate, callbacks=[val_cb],
        args=TrainingArguments(**targs))

    PROG.start("training", f"{len(train_feats)} examples × {P['epochs']} epoch(s)" + (f", resumed at step {resume_step}" if resume else ""))
    trainer.train(resume_from_checkpoint=resume)
    LOSS_LOG = trainer.state.log_history
    PROG.done("training", final_loss=round(LOSS_LOG[-1].get("train_loss", float("nan")), 4))

    set_peft_model_state_dict(model, val_cb.best_state)          # keep the best validation checkpoint
    best = min(val_cb.curve, key=lambda r: r["val_CER"])
    print(f"Using the best validation checkpoint: step {best['step']} (val CER {best['val_CER']:.1%})")
    model.save_pretrained(f"{OUT}/adapter"); tok.save_pretrained(f"{OUT}/adapter")
    json.dump({"best_step": int(best["step"]), "log_history": LOSS_LOG}, open(DONE_PATH, "w"), default=str)
    shutil.rmtree(CKPT_DIR, ignore_errors=True)                  # resume checkpoints no longer needed
    print("Adapter saved to", f"{OUT}/adapter")

In [ ]:
# Learning curve: validation error vs. examples seen, and training loss
curve = pd.DataFrame(val_cb.curve)
loss = [(r["step"] * eff_batch, r["loss"]) for r in LOSS_LOG if "loss" in r]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for c in [c for c in curve if c.startswith("from ")]:
    axes[0].plot(curve.examples_seen, curve[c], marker="o", label=c)
axes[0].plot(curve.examples_seen, curve.val_CER, "k--", lw=2, label="all")
axes[0].set(xlabel="training examples seen", ylabel="validation CER (lower is better)", title="Validation error while training")
axes[0].set_ylim(bottom=0); axes[0].legend(fontsize=8)
if loss:
    axes[1].plot(*zip(*loss)); axes[1].set(xlabel="training examples seen", ylabel="loss", title="Training loss")
plt.tight_layout(); plt.savefig(f"{OUT}/learning_curve.png", dpi=150); plt.show()
display(curve.set_index("examples_seen").drop(columns="step").style.format("{:.1%}"))

## 13. After tuning: the same test sets

In [ ]:
model.eval(); model.config.use_cache = True
PROG.start("tuned eval", f"{n_eval} prompts")
tuned_rows = run_eval("tuned", TEST)
PROG.done("tuned eval", no_answer=f"{np.mean([r['no_answer'] for r in tuned_rows]):.1%}")

## 14. Results

### Error by direction, before → after

In [ ]:
def before_after(split, cols=("CER", "CER loose", "WER", "exact")):
    b, t = direction_table(base_rows, split), direction_table(tuned_rows, split)
    out = pd.DataFrame(index=b.index)
    for c in cols:
        out[c] = [f"{x:.1%} → {y:.1%}" for x, y in zip(b[c], t[c])]
    out["CER change"] = [f"{y - x:+.1%}" for x, y in zip(b["CER"], t["CER"])]
    return out

for split in TEST:
    print(f"\n=== {split} ===")
    display(before_after(split))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, split in zip(axes, TEST):
    b, t = direction_table(base_rows, split), direction_table(tuned_rows, split)
    x = np.arange(len(b))
    ax.bar(x - 0.2, b["CER"], 0.4, label="before", color="#B8C0CC")
    ax.bar(x + 0.2, t["CER"], 0.4, label="after", color="#2E6F9E")
    ax.set_xticks(x, b.index, rotation=30, ha="right", fontsize=8); ax.set_title(split)
axes[0].set_ylabel("character error rate (lower is better)"); axes[0].legend()
plt.suptitle(f"{M['id']}: transliteration error before vs after LoRA ({SIZE})"); plt.tight_layout()
plt.savefig(f"{OUT}/before_after.png", dpi=150); plt.show()

### What kinds of words improve

Word accuracy on words that contain each feature, in the target script. This tells you *what* the model gets wrong, not just how often.

In [ ]:
for split in TEST:
    b, t = phenomena_table(base_rows, split), phenomena_table(tuned_rows, split)
    tab = pd.DataFrame({"before": b, "after": t})
    tab["change"] = tab["after"] - tab["before"]
    tab.index = pd.MultiIndex.from_tuples(tab.index, names=["target script", "words with"])
    print(f"\n=== {split} ===")
    display(tab.style.format("{:.1%}").format("{:+.1%}", subset=["change"])
              .background_gradient(subset=["after"], cmap="Greens", vmin=0, vmax=1))

### Form preserved (rule-based check)

For each matla: does the transliteration still carry the correct radif and qafiya, as extracted by rule from the model's own output?

In [ ]:
rows = []
for split in TEST:
    fb, ft = form_rates(base_rows, split), form_rates(tuned_rows, split)
    for k, name in (("radif_kept", "radif preserved"), ("qafiya_kept", "qafiya preserved")):
        rows.append({"split": split, "check": name, "before": fb.get(k), "after": ft.get(k)})
display(pd.DataFrame(rows).set_index(["split", "check"]).style.format("{:.1%}", na_rep="–"))

### Side by side: the same verses, before and after (words that differ from Rekhta's text in ⟦ ⟧)

In [ ]:
def mark(gold, pred):
    # Mark predicted words that differ from Rekhta's text, line by line (⟦∅⟧ = word missing).
    glines, plines = strict(gold).splitlines(), strict(pred).splitlines()
    if len(plines) != len(glines):                 # line break lost or extra: align as one sequence
        glines, plines = [" ".join(glines)], [" ".join(plines)]
    marked = []
    for gl, pl in zip(glines, plines):
        gw, pw, out = gl.split(), pl.split(), []
        for tag, i1, i2, j1, j2 in SequenceMatcher(None, gw, pw, autojunk=False).get_opcodes():
            seg = " ".join(pw[j1:j2])
            out.append(seg if tag == "equal" else (f"⟦{seg}⟧" if seg else "⟦∅⟧"))
        marked.append(" ".join(x for x in out if x))
    return " / ".join(marked)

bmap = {(r["split"], r["id"]): r for r in base_rows}
shown = defaultdict(int)
for r in tuned_rows:
    d = (r["src"], r["tgt"])
    if r["split"] != "test_unseen" or shown[d] >= 1 or r["sher"] != 0:
        continue
    b = bmap[(r["split"], r["id"])]
    shown[d] += 1
    print(f"--- {DIR_LABEL[d]} · {r['poet']}")
    print(f"Rekhta : {r['gold'].replace(chr(10), ' / ')}")
    print(f"before : {mark(r['gold'], b['pred']) or '(no answer)'}")
    print(f"after  : {mark(r['gold'], r['pred']) or '(no answer)'}\n")

## 15. Verdict

In [ ]:
def check(ok, msg):
    print(("✅ " if ok else "⚠️  ") + msg); return ok

ok_all = True
for split in TEST:
    b, t = direction_table(base_rows, split), direction_table(tuned_rows, split)
    ok_all &= check(t.loc["all directions", "CER"] < b.loc["all directions", "CER"],
                    f"{split}: CER {b.loc['all directions', 'CER']:.1%} → {t.loc['all directions', 'CER']:.1%}, "
                    f"exact shers {b.loc['all directions', 'exact']:.1%} → {t.loc['all directions', 'exact']:.1%}")
    worse = [d for d in b.index[:-1] if t.loc[d, "CER"] > b.loc[d, "CER"] + 0.01]
    ok_all &= check(not worse, f"{split}: every direction improves" + (f" (worse: {', '.join(worse)})" if worse else ""))
na = np.mean([r["no_answer"] for r in tuned_rows])
ok_all &= check(na < 0.02, f"tuned model always answers: no-answer rate {na:.1%}")
ws = np.mean([r["wrong_script"] for r in tuned_rows])
ok_all &= check(ws < 0.02, f"tuned model answers in the requested script: wrong-script rate {ws:.1%}")
first, bestv = val_cb.curve[0]["val_CER"], min(r["val_CER"] for r in val_cb.curve)
ok_all &= check(bestv < first, f"validation error falls with more data: {first:.1%} at 0 examples → best {bestv:.1%}")
fr = form_rates(tuned_rows, "test_unseen").get("radif_kept", float("nan"))
check(fr > 0.8, f"unseen poets: radif preserved in {fr:.0%} of transliterated matlas")
print("\n" + ("Works end to end, and fine-tuning reduces transliteration error on unseen poems and poets. "
              "Next: SIZE='medium' or 'large' and compare the validation curves." if ok_all
              else "Some checks need a look: see the tables above before scaling up."))

## 16. Summary and saved files

In [ ]:
summary = {"model": M["id"], "quantization": QUANT, "size": SIZE, "preset": P, "think_prefill": THINK_PREFILL,
           "train_examples": len(train_feats), "best_val_step": best["step"],
           "by_direction": {s: {"before": direction_table(base_rows, s)[["CER", "WER", "exact"]].to_dict(),
                                "after": direction_table(tuned_rows, s)[["CER", "WER", "exact"]].to_dict()} for s in TEST},
           "form_preserved": {s: {"before": form_rates(base_rows, s), "after": form_rates(tuned_rows, s)} for s in TEST},
           "validation_curve": val_cb.curve, "timing_minutes": PROG.table().set_index("stage")["minutes"].to_dict()}
with open(f"{OUT}/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
with open(f"{OUT}/results.md", "w", encoding="utf-8") as f:
    f.write(f"# {M['id']} · {SIZE} · transliteration\n\n")
    for s in TEST:
        f.write(f"## {s}\n\n" + before_after(s).to_markdown() + "\n\n")
display(PROG.table())
print("Saved in", OUT)
for fn in sorted(os.listdir(OUT)):
    print("  ", fn)

## Next

* **More data:** `SIZE = "medium"` or `"large"` (or set `N_TRAIN`). Each run writes `validation_curve.csv` and `summary.json` in its own folder, so curves can be compared across sizes.
* **Another model:** `MODEL = "qwen2.5-7b"` as a general-model comparison point.
* **Your own corpus:** point `CORPUS_JSONL` at an export. Splits are hashed by poem id, so adding poems never reshuffles existing ones.
* **Phase 2:** with radif, qafiya and meter as rule-based scorers, constrained composition (*write a sher in this zameen*) can be trained with verifiable rewards.